In [1]:
import pandas as pd
import numpy as np
import psycopg

In [2]:
conn = psycopg.connect("dbname=dailyedge_development")

print("Connected to dailyedge_development")

Connected to dailyedge_development


In [3]:
query = """
SELECT
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    COUNT(*) AS total_rows
FROM CANDLES;
"""

db_info = pd.read_sql(query, conn)
db_info

/tmp/ipykernel_41349/1321250083.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_info = pd.read_sql(query, conn)


,first_timestamp,last_timestamp,total_rows
0,2008-12-11 01:38:00,2026-08-28 15:59:00,5943354


In [4]:
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM CANDLES
WHERE timestamp >= '2024-09-02 08:30:00'
  AND timestamp <= '2026-08-21 15:15:00'
  AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
ORDER BY timestamp;
"""

candles = pd.read_sql(query, conn)

print("Rows:", len(candles))
print("First timestamp:", candles["timestamp"].min())
print("Last timestamp:", candles["timestamp"].max())

/tmp/ipykernel_41349/3792214476.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql(query, conn)


Rows: 202598
First timestamp: 2024-09-02 08:30:00
Last timestamp: 2026-08-21 15:15:00


In [5]:
candles["Date"] = candles["timestamp"].dt.date
candles["Day"] = candles["timestamp"].dt.day_name()

session_counts = (
    candles.groupby(["Date", "Day"])
    .size()
    .reset_index(name="Candles")
)

print("Sessions:", len(session_counts))
print()
print(session_counts["Day"].value_counts().sort_index())

Sessions: 508

Day
Friday       101
Monday       103
Thursday     100
Tuesday      103
Wednesday    101
Name: count, dtype: int64


In [6]:
def evaluate_raw_cleanliness(session, stop=35, target=70):
    session = session.sort_values("timestamp").reset_index(drop=True)

    opening_price = session.iloc[0]["open"]

    upper_target = opening_price + target
    lower_target = opening_price - target

    upper_hits = session.index[session["high"] >= upper_target]
    lower_hits = session.index[session["low"] <= lower_target]

    if len(upper_hits) == 0 and len(lower_hits) == 0:
        return "Neither", None

    if len(upper_hits) == 0:
        direction = "Short"
        target_idx = lower_hits[0]
    elif len(lower_hits) == 0:
        direction = "Long"
        target_idx = upper_hits[0]
    else:
        first_upper = upper_hits[0]
        first_lower = lower_hits[0]

        if first_upper == first_lower:
            return "Unknown", "Unknown"

        if first_upper < first_lower:
            direction = "Long"
            target_idx = first_upper
        else:
            direction = "Short"
            target_idx = first_lower

    before_target = session.loc[:target_idx]

    if direction == "Long":
        adverse_level = opening_price - stop

        adverse_hits = before_target.index[
            before_target["low"] <= adverse_level
        ]

    else:
        adverse_level = opening_price + stop

        adverse_hits = before_target.index[
            before_target["high"] >= adverse_level
        ]

    if len(adverse_hits) == 0:
        cleanliness = "Clean"
    else:
        first_adverse = adverse_hits[0]

        if first_adverse == target_idx:
            cleanliness = "Unknown"
        else:
            cleanliness = "Not Clean"

    return direction, cleanliness

In [7]:
stop_target_pairs = [
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 80),
    (50, 100),
    (65, 100),
    (75, 100),
    (75, 150),
    (100, 80),
    (100, 50),
    (100, 25),
    (100, 100),
    (100, 150)
]

all_raw_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        direction, cleanliness = evaluate_raw_cleanliness(
            session,
            stop=stop,
            target=target
        )

        all_raw_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Direction": direction,
            "Cleanliness": cleanliness
        })

all_raw_results = pd.DataFrame(all_raw_results)

print("Rows:", len(all_raw_results))
print()
print(
    all_raw_results
    .groupby(["Stop", "Target"])["Cleanliness"]
    .value_counts(dropna=False)
)

Rows: 7112

Stop  Target  Cleanliness
15    25      Clean          334
              Not Clean       94
              Unknown         79
              NaN              1
25    50      Clean          338
              Not Clean      155
              Unknown         10
              NaN              5
35    70      Clean          340
              Not Clean      153
              NaN             15
50    70      Clean          409
              Not Clean       83
              NaN             15
              Unknown          1
      80      Clean          375
              Not Clean      115
              NaN             18
      100     Clean          327
              Not Clean      147
              NaN             34
65    100     Clean          385
              Not Clean       89
              NaN             34
75    100     Clean          416
              Not Clean       58
              NaN             34
      150     Clean          319
              NaN            100
     

In [8]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

resolved_raw = all_raw_results[
    all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
].copy()

resolved_raw["Clean"] = resolved_raw["Cleanliness"] == "Clean"

raw_weekday_summary = (
    resolved_raw
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Clean", "size"),
        Clean=("Clean", "sum")
    )
    .reset_index()
)

raw_weekday_summary["Clean Rate"] = (
    raw_weekday_summary["Clean"]
    / raw_weekday_summary["Resolved"]
    * 100
)

raw_cleanliness_table = (
    raw_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Clean Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved = (
    all_raw_results[
        ~all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

raw_cleanliness_table["Unresolved"] = unresolved

raw_cleanliness_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       79.76    75.28      77.78     77.38   80.25          80
25   50       66.67    67.33      73.00     71.88   63.92          15
35   70       66.32    69.90      72.28     69.39   66.67          15
50   70       81.05    84.47      83.17     83.67   83.16          16
     80       80.00    76.70      74.26     76.04   75.79          18
     100      71.43    65.66      67.35     69.15   71.74          34
65   100      85.71    76.77      81.63     78.72   83.70          34
75   100      92.31    81.82      88.78     87.23   89.13          34
     150      84.29    73.26      80.46     75.00   79.01         100
100  25      100.00   100.00     100.00    100.00  100.00          23
     50      100.00   100.00     100.00    100.00  100.00           7
     80      100.00   100.00     100.00    100.00  100.00          18
     100     100.00   100.00     100.00    100.00  100.00          34
     150      90.00    89.53      91.95     85.71   90.12         100

In [9]:
def evaluate_continuous_trail(
    rth,
    opening_price,
    target_distance,
    trail_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)
            new_stop = new_highest - trail_distance

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            # Low first
            if old_stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif new_stop_hit:
                high_first = "Failure"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if {low_first, high_first} == {"Continue", "Failure"}:
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if low_first == "Continue" and high_first == "Continue":
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + trail_distance

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            # High first
            if old_stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif new_stop_hit:
                low_first = "Failure"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if {high_first, low_first} == {"Continue", "Failure"}:
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if high_first == "Continue" and low_first == "Continue":
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [10]:
def get_first_target(rth, opening_price, target_distance):
    upper_target = opening_price + target_distance
    lower_target = opening_price - target_distance

    target_hits = rth[
        (rth["high"] >= upper_target) |
        (rth["low"] <= lower_target)
    ]

    if target_hits.empty:
        return "Neither", None

    first_hit = target_hits.iloc[0]

    hit_upper = first_hit["high"] >= upper_target
    hit_lower = first_hit["low"] <= lower_target

    if hit_upper and hit_lower:
        return "Ambiguous", first_hit["timestamp"]

    result = "Long" if hit_upper else "Short"

    return result, first_hit["timestamp"]

In [11]:
all_trail_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_continuous_trail(
            session,
            opening_price,
            target_distance=target,
            trail_distance=stop
        )

        all_trail_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_trail_results = pd.DataFrame(all_trail_results)

print("Rows:", len(all_trail_results))
print()
print(
    all_trail_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 7112

Stop  Target  Outcome  
15    25      Success      285
              Failure      133
              Unknown       67
              Ambiguous     22
              Neither        1
25    50      Failure      284
              Success      201
              Unknown       16
              Neither        5
              Ambiguous      2
35    70      Failure      321
              Success      168
              Neither       15
              Unknown        4
50    70      Success      273
              Failure      219
              Neither       15
              Unknown        1
      80      Failure      257
              Success      228
              Neither       18
              Unknown        5
      100     Failure      314
              Success      156
              Neither       34
              Unknown        4
65    100     Success      238
              Failure      232
              Neither       34
              Unknown        4
75    100     Success      274
   

In [12]:
resolved_trail = all_trail_results[
    all_trail_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_trail["Success"] = resolved_trail["Outcome"] == "Success"

trail_weekday_summary = (
    resolved_trail
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

trail_weekday_summary["Survival Rate"] = (
    trail_weekday_summary["Success"]
    / trail_weekday_summary["Resolved"]
    * 100
)

trail_survival_table = (
    trail_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_trail = (
    all_trail_results[
        ~all_trail_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

trail_survival_table["Unresolved"] = unresolved_trail

trail_survival_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       68.35    65.52      68.18     69.05   70.00          90
25   50       33.67    40.40      45.45     45.16   42.71          23
35   70       36.17    32.67      37.00     31.63   34.38          19
50   70       60.00    53.40      54.46     52.04   57.89          16
     80       55.79    46.00      44.44     43.75   45.26          23
     100      42.22    27.55      28.87     31.91   36.26          38
65   100      61.54    42.86      49.48     52.69   47.25          38
75   100      70.33    53.06      56.70     56.99   54.95          38
     150      44.29    38.37      32.18     46.43   35.80         100
100  25      100.00   100.00     100.00    100.00  100.00          23
     50      100.00   100.00     100.00    100.00  100.00           7
     80       90.53    90.20      92.08     89.58   86.32          19
     100      82.42    78.57      78.57     75.53   72.83          35
     150      57.14    58.14      52.87     63.10   55.56         100

In [13]:
def evaluate_one_move_breakeven(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        initial_stop = opening_price - stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = high >= threshold

            # Low first
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                if low <= opening_price:
                    high_first = "Failure"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        initial_stop = opening_price + stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                if high >= opening_price:
                    low_first = "Failure"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [14]:
all_be_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_one_move_breakeven(
            session,
            opening_price,
            target_distance=target,
            stop_distance=stop
        )

        all_be_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_be_results = pd.DataFrame(all_be_results)

print("Rows:", len(all_be_results))
print()
print(
    all_be_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 7112

Stop  Target  Outcome  
15    25      Success      336
              Failure       81
              Unknown       68
              Ambiguous     22
              Neither        1
25    50      Success      317
              Failure      173
              Unknown       11
              Neither        5
              Ambiguous      2
35    70      Success      310
              Failure      181
              Neither       15
              Unknown        2
50    70      Success      408
              Failure       83
              Neither       15
              Unknown        2
      80      Success      366
              Failure      122
              Neither       18
              Unknown        2
      100     Success      299
              Failure      173
              Neither       34
              Unknown        2
65    100     Success      361
              Failure      111
              Neither       34
              Unknown        2
75    100     Success      400
   

In [15]:
resolved_be = all_be_results[
    all_be_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_be["Success"] = resolved_be["Outcome"] == "Success"

be_weekday_summary = (
    resolved_be
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

be_weekday_summary["Survival Rate"] = (
    be_weekday_summary["Success"]
    / be_weekday_summary["Resolved"]
    * 100
)

be_survival_table = (
    be_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_be = (
    all_be_results[
        ~all_be_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

be_survival_table["Unresolved"] = unresolved_be

be_survival_table.round(2)


Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       79.01    77.91      78.16     82.14   86.08          91
25   50       60.82    65.35      66.00     66.67   64.58          18
35   70       62.77    60.19      63.37     65.31   64.21          17
50   70       84.04    82.52      85.15     84.69   78.95          17
     80       78.95    73.53      77.00     76.04   69.47          20
     100      72.53    60.20      61.86     63.83   58.70          36
65   100      83.52    74.49      74.23     74.47   76.09          36
75   100      85.71    84.69      83.67     81.91   86.96          35
     150      67.14    70.93      60.92     70.24   71.60         100
100  25      100.00   100.00     100.00    100.00  100.00          23
     50      100.00   100.00     100.00    100.00  100.00           7
     80      100.00   100.00     100.00    100.00  100.00          18
     100     100.00   100.00     100.00    100.00  100.00          34
     150      77.14    83.72      70.11     80.95   85.19         100

In [16]:
# Load evaluated First-to-100 predictions

predictions = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")

predictions["Date"] = pd.to_datetime(predictions["Date"])

print("Prediction rows:", len(predictions))
print("Date range:", predictions["Date"].min(), "to", predictions["Date"].max())

predictions[["Date", "Day", "Bias", "Result", "Correct"]].head()

Prediction rows: 512
Date range: 2024-09-02 00:00:00 to 2026-08-28 00:00:00


,Date,Day,Bias,Result,Correct
0,2024-09-02,Monday,Long,Invalid,NaN
1,2024-09-03,Tuesday,Short,Short,True
2,2024-09-04,Wednesday,Short,Long,False
3,2024-09-05,Thursday,Short,Long,False
4,2024-09-06,Friday,Long,Short,False


In [17]:
# Check date overlap between predictions and cleanliness results

prediction_dates = set(
    predictions["Date"].dropna().dt.date
)

cleanliness_dates = set(
    pd.to_datetime(all_be_results["Date"]).dropna().dt.date
)

matched_dates = prediction_dates & cleanliness_dates
prediction_only = prediction_dates - cleanliness_dates
cleanliness_only = cleanliness_dates - prediction_dates

print("Prediction dates:", len(prediction_dates))
print("Cleanliness dates:", len(cleanliness_dates))
print("Matched dates:", len(matched_dates))

print("\nPrediction-only dates:", sorted(prediction_only))
print("\nCleanliness-only dates:", sorted(cleanliness_only))

Prediction dates: 512
Cleanliness dates: 508
Matched dates: 505

Prediction-only dates: [datetime.date(2025, 1, 9), datetime.date(2026, 4, 3), datetime.date(2026, 8, 24), datetime.date(2026, 8, 25), datetime.date(2026, 8, 26), datetime.date(2026, 8, 27), datetime.date(2026, 8, 28)]

Cleanliness-only dates: [datetime.date(2025, 1, 20), datetime.date(2025, 3, 4), datetime.date(2026, 4, 24)]


In [18]:
# Prediction accuracy population for matched dates

matched_predictions = predictions[
    predictions["Date"].dt.date.isin(matched_dates)
].copy()

print("Matched predictions:", len(matched_predictions))

print("\nCorrect distribution:")
print(matched_predictions["Correct"].value_counts(dropna=False))

print("\nResult distribution:")
print(matched_predictions["Result"].value_counts(dropna=False))

Matched predictions: 505

Correct distribution:
Correct
True     259
False    223
NaN       23
Name: count, dtype: int64

Result distribution:
Result
Short      250
Long       231
Invalid     18
Neither      6
Name: count, dtype: int64


In [19]:
# Merge resolved prediction accuracy with BE + Add cleanliness results

resolved_predictions = matched_predictions[
    matched_predictions["Correct"].notna()
].copy()

# Standardize Date dtype before merging
all_be_results["Date"] = pd.to_datetime(all_be_results["Date"])

be_accuracy = all_be_results.merge(
    resolved_predictions[["Date", "Correct"]],
    on="Date",
    how="inner"
)

print("Resolved prediction dates:", len(resolved_predictions))
print("Merged rows:", len(be_accuracy))
print("Unique merged dates:", be_accuracy["Date"].nunique())

be_accuracy.head()

Resolved prediction dates: 482
Merged rows: 6748
Unique merged dates: 482


,Date,Day,Stop,Target,Outcome,Correct
0,2024-09-03,Tuesday,15,25,Success,True
1,2024-09-04,Wednesday,15,25,Failure,False
2,2024-09-05,Thursday,15,25,Success,False
3,2024-09-06,Friday,15,25,Success,False
4,2024-09-09,Monday,15,25,Success,True


In [20]:
# BE + Add success rate by weekday — correct predictions only

correct_be = be_accuracy[
    (be_accuracy["Correct"] == True) &
    (be_accuracy["Outcome"].isin(["Success", "Failure"]))
].copy()

correct_success_table = (
    correct_be
    .assign(Success=correct_be["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Success"]
    .mean()
    .mul(100)
    .unstack("Day")
)

correct_success_table = correct_success_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

correct_success_table.index = [
    f"{stop}/{target}"
    for stop, target in correct_success_table.index
]

correct_success_table.index.name = "Pair"

correct_success_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,75.68,80.95,83.33,91.18,87.80
25/50,64.58,60.38,68.97,68.42,72.22
35/70,67.35,60.00,69.49,57.50,70.37
50/70,86.00,81.82,86.44,80.00,75.93
50/80,84.00,75.93,75.86,69.23,66.67
50/100,73.47,62.26,60.71,50.00,57.69
65/100,85.71,77.36,71.43,63.16,75.00
75/100,85.71,86.79,84.21,76.32,84.62
75/150,69.23,79.55,63.27,60.00,71.11


In [21]:
# Actual hit rate by weekday:
# prediction must be correct AND BE + Add trade must succeed

weekday_accuracy = (
    resolved_predictions
    .groupby("Day")["Correct"]
    .mean()
)

actual_hit_rate = correct_success_table.copy()

for day in actual_hit_rate.columns:
    actual_hit_rate[day] = (
        actual_hit_rate[day] * weekday_accuracy[day]
    )

actual_hit_rate.index.name = "Pair"

print("Prediction accuracy used:")
display((weekday_accuracy * 100).round(2))

print("\nActual hit rate (%):")
display(actual_hit_rate.round(2))

Prediction accuracy used:


Day
Friday       59.14
Monday       54.35
Thursday     41.67
Tuesday      54.46
Wednesday     59.0
Name: Correct, dtype: object


Actual hit rate (%):


Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,41.13,44.08,49.17,37.99,51.93
25/50,35.10,32.88,40.69,28.51,42.71
35/70,36.60,32.67,41.00,23.96,41.62
50/70,46.74,44.55,51.00,33.33,44.90
50/80,45.65,41.35,44.76,28.85,39.43
50/100,39.93,33.91,35.82,20.83,34.12
65/100,46.58,42.13,42.14,26.32,44.35
75/100,46.58,47.26,49.68,31.80,50.04
75/150,37.63,43.32,37.33,25.00,42.05


In [22]:
# Continuous trail success rate by weekday — correct predictions only

all_trail_results["Date"] = pd.to_datetime(all_trail_results["Date"])

trail_accuracy = all_trail_results.merge(
    resolved_predictions[["Date", "Correct"]],
    on="Date",
    how="inner"
)

correct_trail = trail_accuracy[
    (trail_accuracy["Correct"] == True) &
    (trail_accuracy["Outcome"].isin(["Success", "Failure"]))
].copy()

trail_success_table = (
    correct_trail
    .assign(Success=correct_trail["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Success"]
    .mean()
    .mul(100)
    .unstack("Day")
)

trail_success_table = trail_success_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

trail_success_table.index = [
    f"{stop}/{target}"
    for stop, target in trail_success_table.index
]

trail_success_table.index.name = "Pair"

trail_success_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,67.57,71.11,72.22,88.24,64.29
25/50,37.50,43.14,50.88,44.44,53.70
35/70,44.90,25.93,48.28,30.00,41.82
50/70,66.00,52.73,62.71,47.50,62.96
50/80,60.00,47.17,45.61,38.46,48.15
50/100,41.67,28.30,30.36,23.68,42.31
65/100,61.22,45.28,50.00,43.24,53.85
75/100,73.47,54.72,55.36,45.95,57.69
75/150,48.72,50.00,34.69,37.14,31.11


In [23]:
all_be_results["Outcome"].value_counts(dropna=False)

Outcome
Success      5349
Failure      1211
Neither       414
Unknown        90
Ambiguous      48
Name: count, dtype: int64

In [24]:
def evaluate_one_move_breakeven_no_add(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        stop = opening_price - stop_distance
        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = high >= threshold

            # Low first
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                if low <= opening_price:
                    high_first = "Breakeven"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first in ["Success", "Failure", "Breakeven"]:
                    return low_first

            if low_first != high_first and {
                low_first,
                high_first
            } <= {"Success", "Failure", "Breakeven"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        stop = opening_price + stop_distance
        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                if high >= opening_price:
                    low_first = "Breakeven"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first in ["Success", "Failure", "Breakeven"]:
                    return high_first

            if high_first != low_first and {
                high_first,
                low_first
            } <= {"Success", "Failure", "Breakeven"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [25]:
all_be_no_add_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_one_move_breakeven_no_add(
            session,
            opening_price,
            target_distance=target,
            stop_distance=stop
        )

        all_be_no_add_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_be_no_add_results = pd.DataFrame(all_be_no_add_results)

print("Rows:", len(all_be_no_add_results))
print()
print(
    all_be_no_add_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 7112

Stop  Target  Outcome  
15    25      Success      336
              Unknown       97
              Breakeven     47
              Ambiguous     22
              Failure        5
              Neither        1
25    50      Success      317
              Breakeven    116
              Failure       41
              Unknown       27
              Neither        5
              Ambiguous      2
35    70      Success      310
              Breakeven    119
              Failure       56
              Neither       15
              Unknown        8
50    70      Success      408
              Breakeven     59
              Failure       23
              Neither       15
              Unknown        3
      80      Success      366
              Breakeven     73
              Failure       46
              Neither       18
              Unknown        5
      100     Success      299
              Breakeven    105
              Failure       64
              Neither       34
   

In [26]:
def evaluate_predicted_one_move_breakeven(
    rth,
    opening_price,
    direction,
    target_distance,
    stop_distance
):
    # -------------------------
    # LONG
    # -------------------------

    if direction == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance
        stop = opening_price - stop_distance

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = high >= threshold

            # Stop definitely hit before threshold/target
            if stop_hit and not threshold_hit and not target_hit:
                return "Failure"

            # Target reached without stop also being touched
            if target_hit and not stop_hit:
                return "Success"

            # Both sides touched in same candle: order unknown
            if stop_hit and (threshold_hit or target_hit):
                return "Unknown"

            # Threshold reached: move stop to breakeven
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

    # -------------------------
    # SHORT
    # -------------------------

    elif direction == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance
        stop = opening_price + stop_distance

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = low <= threshold

            # Stop definitely hit before threshold/target
            if stop_hit and not threshold_hit and not target_hit:
                return "Failure"

            # Target reached without stop also being touched
            if target_hit and not stop_hit:
                return "Success"

            # Both sides touched in same candle: order unknown
            if stop_hit and (threshold_hit or target_hit):
                return "Unknown"

            # Threshold reached: move stop to breakeven
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

    else:
        return "Invalid"

    return "Neither"

In [27]:
predicted_be_results = []

for stop, target in stop_target_pairs:
    for _, prediction in resolved_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Bias"]

        session = candles[
            candles["Date"] == date.date()
        ].sort_values("timestamp").reset_index(drop=True)

        if session.empty:
            continue

        opening_price = session.iloc[0]["open"]

        outcome = evaluate_predicted_one_move_breakeven(
            session,
            opening_price,
            direction=direction,
            target_distance=target,
            stop_distance=stop
        )

        predicted_be_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Correct": prediction["Correct"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

predicted_be_results = pd.DataFrame(predicted_be_results)

print("Rows:", len(predicted_be_results))
print("Unique dates:", predicted_be_results["Date"].nunique())

print()
print(
    predicted_be_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 6748
Unique dates: 482

Stop  Target  Outcome  
15    25      Failure      183
              Success      156
              Unknown      107
              Breakeven     36
25    50      Failure      224
              Success      153
              Breakeven     83
              Unknown       22
35    70      Failure      222
              Success      146
              Breakeven    108
              Unknown        6
50    70      Failure      216
              Success      199
              Breakeven     65
              Unknown        2
      80      Failure      216
              Success      178
              Breakeven     84
              Unknown        4
      100     Failure      216
              Success      142
              Breakeven    120
              Unknown        4
65    100     Failure      227
              Success      173
              Breakeven     80
              Unknown        2
75    100     Failure      221
              Success      197
              Br

In [28]:
# Actual one-time BE hit rate by weekday
# Breakeven / Unknown / Neither excluded

resolved_predicted_be = predicted_be_results[
    predicted_be_results["Outcome"].isin(["Success", "Failure"])
].copy()

be_hit_rate_table = (
    resolved_predicted_be
    .assign(Hit=resolved_predicted_be["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Hit"]
    .mean()
    .mul(100)
    .unstack("Day")
)

be_hit_rate_table = be_hit_rate_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

be_hit_rate_table.index = [
    f"{stop}/{target}"
    for stop, target in be_hit_rate_table.index
]

be_hit_rate_table.index.name = "Pair"

be_hit_rate_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,43.75,43.66,54.55,40.30,46.67
25/50,39.06,40.79,50.60,29.27,43.06
35/70,42.03,37.97,53.42,26.39,38.67
50/70,52.50,48.28,57.65,34.94,46.25
50/80,51.28,45.78,53.85,31.65,43.42
50/100,47.22,40.00,47.06,25.00,39.44
65/100,50.00,44.19,48.05,27.16,47.37
75/100,49.40,47.73,54.65,31.65,51.22
75/150,37.31,41.03,45.07,25.00,43.66


In [29]:
def evaluate_continuous_trail_strict(
    rth,
    opening_price,
    target_distance,
    trail_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            current_stop = highest_price - trail_distance
            if open_price <= current_stop:
                return "Failure"

            # Peak including this candle's own high. Any pullback of
            # trail_distance off THIS peak -- even intra-candle -- is a failure.
            new_highest = max(highest_price, high)
            new_stop = new_highest - trail_distance

            stop_hit = low <= new_stop
            target_hit = high >= target

            if stop_hit and target_hit:
                return "Unknown"
            if target_hit:
                return "Success"
            if stop_hit:
                return "Failure"

            highest_price = new_highest

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            current_stop = lowest_price + trail_distance
            if open_price >= current_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + trail_distance

            stop_hit = high >= new_stop
            target_hit = low <= target

            if stop_hit and target_hit:
                return "Unknown"
            if target_hit:
                return "Success"
            if stop_hit:
                return "Failure"

            lowest_price = new_lowest

    return None

In [30]:
all_strict_trail_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_continuous_trail_strict(
            session,
            opening_price,
            target_distance=target,
            trail_distance=stop
        )

        all_strict_trail_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_strict_trail_results = pd.DataFrame(all_strict_trail_results)

print("Rows:", len(all_strict_trail_results))
print()
print(
    all_strict_trail_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 7112

Stop  Target  Outcome  
15    25      Unknown      302
              Failure      182
              Ambiguous     22
              Neither        1
              Success        1
25    50      Failure      386
              Unknown      105
              Success       10
              Neither        5
              Ambiguous      2
35    70      Failure      426
              Success       36
              Unknown       31
              Neither       15
50    70      Failure      309
              Success      139
              Unknown       45
              Neither       15
      80      Failure      347
              Success      116
              Unknown       27
              Neither       18
      100     Failure      381
              Success       83
              Neither       34
              Unknown       10
65    100     Failure      283
              Success      186
              Neither       34
              Unknown        5
75    100     Success      249
   

In [31]:
resolved_strict_trail = all_strict_trail_results[
    all_strict_trail_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_strict_trail["Success"] = resolved_strict_trail["Outcome"] == "Success"

strict_trail_weekday_summary = (
    resolved_strict_trail
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

strict_trail_weekday_summary["Survival Rate"] = (
    strict_trail_weekday_summary["Success"]
    / strict_trail_weekday_summary["Resolved"]
    * 100
)

strict_trail_survival_table = (
    strict_trail_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_strict_trail = (
    all_strict_trail_results[
        ~all_strict_trail_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

strict_trail_survival_table["Unresolved"] = unresolved_strict_trail

strict_trail_survival_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25        2.56     0.00       0.00      0.00    0.00         325
25   50        7.06     1.25       1.27      1.35    1.28         112
35   70        8.89     9.78       8.42      6.38    5.49          46
50   70       32.56    30.43      31.52     28.57   32.18          60
     80       29.55    25.51      25.77     20.88   23.60          45
     100      23.33    14.43      15.62     17.20   19.32          44
65   100      48.35    30.30      42.71     39.78   37.78          39
75   100      62.22    48.48      53.12     49.46   53.33          40
     150      39.13    34.52      30.23     42.86   33.33         104
100  25      100.00   100.00     100.00    100.00  100.00          30
     50      100.00   100.00     100.00    100.00  100.00          18
     80       89.13    89.00      91.58     88.42   85.11          32
     100      79.12    75.51      77.17     74.19   70.79          45
     150      57.14    56.98      49.43     63.10   52.50         101